# Memory Optimization Helper for Large Datasets

This notebook provides helper functions to work with large CSV files (like user_logs.csv - 28GB) without running out of memory.

## Strategies:
1. **Chunked Reading**: Process data in smaller chunks
2. **Column Selection**: Only load needed columns
3. **Data Type Optimization**: Use smaller data types
4. **Aggregation**: Aggregate data before loading fully

In [1]:
# Install all required libraries
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm tqdm

In [2]:
import pandas as pd
import numpy as np
import gc
from tqdm import tqdm

## 1. Check Available Memory

In [3]:
def check_memory():
    """Check current memory usage"""
    import psutil
    mem = psutil.virtual_memory()
    print(f"Total Memory: {mem.total / (1024**3):.2f} GB")
    print(f"Available Memory: {mem.available / (1024**3):.2f} GB")
    print(f"Used Memory: {mem.used / (1024**3):.2f} GB")
    print(f"Memory Usage: {mem.percent}%")
    return mem

check_memory()

Total Memory: 7.65 GB
Available Memory: 7.06 GB
Used Memory: 0.59 GB
Memory Usage: 7.7%


svmem(total=8217858048, available=7583576064, percent=7.7, used=634281984, free=7554060288, active=372391936, inactive=86376448, buffers=11784192, cached=181538816, shared=0, slab=72798208)

## 2. Reduce Memory Usage Function

In [4]:
def reduce_mem_usage(df, verbose=True):
    """
    Reduce memory usage by optimizing data types
    """
    start_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f'Memory usage before optimization: {start_mem:.2f} MB')
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f'Memory usage after optimization: {end_mem:.2f} MB')
        print(f'Decreased by {100 * (start_mem - end_mem) / start_mem:.1f}%')
    
    return df

## 3. Read Large CSV in Chunks with Aggregation

In [5]:
def read_user_logs_aggregated(file_path='data/user_logs_v2.csv', chunksize=1000000):
    """
    Read user_logs in chunks and aggregate by user (msno)
    This reduces 28GB to a manageable size
    """
    print(f"Reading {file_path} in chunks of {chunksize:,} rows...")
    
    # Initialize aggregated dataframe
    aggregated_data = []
    
    # Read in chunks
    chunk_iter = pd.read_csv(file_path, chunksize=chunksize)
    
    for i, chunk in enumerate(tqdm(chunk_iter, desc="Processing chunks")):
        # Aggregate each chunk by user
        chunk_agg = chunk.groupby('msno').agg({
            'num_25': ['sum', 'mean', 'std'],
            'num_50': ['sum', 'mean', 'std'],
            'num_75': ['sum', 'mean', 'std'],
            'num_985': ['sum', 'mean', 'std'],
            'num_100': ['sum', 'mean', 'std'],
            'num_unq': ['sum', 'mean', 'std'],
            'total_secs': ['sum', 'mean', 'std'],
            'date': ['count', 'min', 'max']  # count = number of active days
        }).reset_index()
        
        # Flatten column names
        chunk_agg.columns = ['_'.join(col).strip('_') for col in chunk_agg.columns.values]
        
        aggregated_data.append(chunk_agg)
        
        # Clear memory
        del chunk
        gc.collect()
        
        if (i + 1) % 5 == 0:
            print(f"  Processed {(i + 1) * chunksize:,} rows...")
            check_memory()
    
    print("\nCombining all chunks...")
    # Combine all aggregated chunks
    df_combined = pd.concat(aggregated_data, ignore_index=True)
    
    # Final aggregation (in case same user appears in multiple chunks)
    print("Final aggregation by user...")
    df_final = df_combined.groupby('msno').agg('sum').reset_index()
    
    # Optimize memory
    df_final = reduce_mem_usage(df_final)
    
    print(f"\n✅ Done! Final shape: {df_final.shape}")
    print(f"Memory usage: {df_final.memory_usage().sum() / 1024**2:.2f} MB")
    
    return df_final

## 4. Example: Process user_logs_v2.csv

In [6]:
# Process user logs (this will take some time but won't crash)
user_logs_features = read_user_logs_aggregated('../data/user_logs_v2.csv', chunksize=500000)

# Display first few rows
user_logs_features.head()

Reading ../data/user_logs_v2.csv in chunks of 500,000 rows...


Processing chunks: 5it [00:05,  1.16s/it]

  Processed 2,500,000 rows...
Total Memory: 7.65 GB
Available Memory: 6.30 GB
Used Memory: 1.36 GB
Memory Usage: 17.7%


Processing chunks: 10it [00:11,  1.17s/it]

  Processed 5,000,000 rows...
Total Memory: 7.65 GB
Available Memory: 5.79 GB
Used Memory: 1.86 GB
Memory Usage: 24.4%


...

  Processed 17,500,000 rows...
Total Memory: 7.65 GB
Available Memory: 3.10 GB
Used Memory: 4.55 GB
Memory Usage: 59.5%


Processing chunks: 37it [00:45,  1.24s/it]



Combining all chunks...
Final aggregation by user...
Memory usage before optimization: 210.55 MB
Memory usage after optimization: 72.64 MB
Decreased by 65.5%

✅ Done! Final shape: (1103894, 25)
Memory usage: 72.64 MB


/usr/local/lib/python3.10/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,msno,num_25_sum,num_25_mean,num_25_std,num_50_sum,num_50_mean,num_50_std,num_75_sum,num_75_mean,num_75_std,...,num_100_std,num_unq_sum,num_unq_mean,num_unq_std,total_secs_sum,total_secs_mean,total_secs_std,date_count,date_min,date_max
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,86,55.84375,13.664062,11,7.000000,4.242188,10,6.332031,1.991211,...,86.6875,530,385.000,91.25000,117907.421875,87228.703125,21513.908203,26,383235964,383236056
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,191,127.00000,17.921875,90,68.687500,9.757812,75,52.156250,12.531250,...,76.1875,885,587.000,73.06250,192527.890625,128383.531250,20928.357422,31,443746914,443746990
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,43,32.00000,8.484375,12,7.000000,5.656250,15,8.500000,6.363281,...,103.9375,468,340.000,80.62500,115411.257812,82404.492188,24008.904297,28,383235950,383236065
3,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,207,132.62500,45.968750,163,99.000000,61.468750,100,62.500000,31.562500,...,65.2500,828,534.500,189.37500,149896.562500,98342.687500,27451.970703,21,302554687,302554759
4,++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk=,105,62.90625,24.968750,24,16.671875,4.800781,39,21.500000,12.132812,...,111.7500,230,139.125,60.46875,116433.250000,71986.484375,23909.009766,29,383235930,383236024


## 5. Save Processed Features

In [7]:
# Save the aggregated features for future use
user_logs_features.to_csv('../data/user_logs_features_aggregated.csv', index=False)
print("✅ Saved to data/user_logs_features_aggregated.csv")

✅ Saved to data/user_logs_features_aggregated.csv


## 6. Alternative: Sample-based Approach

In [8]:
def read_user_logs_sample(file_path='data/user_logs_v2.csv', sample_size=1000000):
    """
    Read a random sample of user_logs for quick analysis
    """
    print(f"Reading random sample of {sample_size:,} rows...")
    
    # Count total rows first
    n = sum(1 for line in open(file_path)) - 1  # subtract header
    print(f"Total rows in file: {n:,}")
    
    # Calculate skip probability
    skip = sorted(np.random.choice(range(1, n+1), n - sample_size, replace=False))
    
    # Read with skipping
    df = pd.read_csv(file_path, skiprows=skip)
    
    print(f"✅ Loaded {len(df):,} rows")
    return reduce_mem_usage(df)

# Example: Load 1 million random rows
# user_logs_sample = read_user_logs_sample('data/user_logs_v2.csv', sample_size=1000000)

## 7. Load Smaller Datasets Efficiently

In [10]:
# Load other datasets with memory optimization
print("Loading train_v2.csv...")
train = pd.read_csv('../data/train_v2.csv')
train = reduce_mem_usage(train)

print("\nLoading members_v3.csv...")
members = pd.read_csv('../data/members_v3.csv')
members = reduce_mem_usage(members)

print("\nLoading transactions_v2.csv...")
transactions = pd.read_csv('../data/transactions_v2.csv')
transactions = reduce_mem_usage(transactions)

print("\n✅ All datasets loaded!")
check_memory()

Loading train_v2.csv...
Memory usage before optimization: 14.82 MB
Memory usage after optimization: 8.33 MB
Decreased by 43.7%

Loading members_v3.csv...
Memory usage before optimization: 309.88 MB
Memory usage after optimization: 154.94 MB
Decreased by 50.0%

Loading transactions_v2.csv...
Memory usage before optimization: 98.26 MB
Memory usage after optimization: 34.12 MB
Decreased by 65.3%

✅ All datasets loaded!
Total Memory: 7.65 GB
Available Memory: 4.32 GB
Used Memory: 3.34 GB
Memory Usage: 43.6%


svmem(total=8217858048, available=4636925952, percent=43.6, used=3580932096, free=4100665344, active=1186824192, inactive=2658070528, buffers=1466368, cached=699482112, shared=16384, slab=74190848)

## 8. Clear Memory When Needed

In [11]:
def clear_memory():
    """Force garbage collection to free memory"""
    gc.collect()
    print("🧹 Memory cleared")
    check_memory()

# Use when needed:
# clear_memory()